# JOINT FITTING BETWEEN FERMI AND VERITAS (IN GAMMAPY) FOR A POINT SOURCE WITH POWER LAW SPECTRUM USING CRAB DATA

Anjana Kaushik Talluri

Note: The code below relies on J. Michael's model converter 

# Import packages

In [1]:
import warnings

warnings.simplefilter("ignore")
import numpy as np

np.seterr(all="ignore")



import matplotlib.pyplot as plt

import astropy.units as u
from threeML import *
from threeML.io.package_data import get_path_of_data_file



import warnings

warnings.simplefilter("ignore")
import numpy as np

np.seterr(all="ignore")
import shutil
from IPython.display import Image, display
import glob
from pathlib import Path
import matplotlib as mpl
from matplotlib import pyplot as plt
from astropy.io import fits as pyfits
import scipy as sp


from threeML import *



12:21:17 WARNING   The naima package is not available. Models that depend on it will not be         ]8;id=687472;file:///home/tobi/sw/astromodels/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=965314;file:///home/tobi/sw/astromodels/astromodels/functions/functions_1D/functions.py#47\47]8;;\
                  available                                                                                        

         WARNING   The GSL library or the pygsl wrapper cannot be loaded. Models that depend on it  ]8;id=686827;file:///home/tobi/sw/astromodels/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=848365;file:///home/tobi/sw/astromodels/astromodels/functions/functions_1D/functions.py#68\68]8;;\
                  will not be available.                                                                           

         WARNING   The ebltable package is not available. Models that depend on it will not be     ]8;id=59863;file:///home/tobi/sw/astromodels/astromodels/functions/functions_1D/absorption.py\absorption.py]8;;\:]8;id=718513;file:///home/tobi/sw/astromodels/astromodels/functions/functions_1D/absorption.py#33\33]8;;\
                  available                                                                                        

         INFO      Starting 3ML!                                                                     ]8;id=505890;file:///home/tobi/sw/threeML/threeML/__init__.py\__init__.py]8;;\:]8;id=261795;file:///home/tobi/sw/threeML/threeML/__init__.py#39\39]8;;\

         WARNING   WARNINGs here are NOT errors                                                      ]8;id=695075;file:///home/tobi/sw/threeML/threeML/__init__.py\__init__.py]8;;\:]8;id=9181;file:///home/tobi/sw/threeML/threeML/__init__.py#40\40]8;;\

...

         WARNING    to disable these messages, turn off start_warning in your config file            ]8;id=464746;file:///home/tobi/sw/threeML/threeML/__init__.py\__init__.py]8;;\:]8;id=515881;file:///home/tobi/sw/threeML/threeML/__init__.py#44\44]8;;\

12:21:17 WARNING   ROOT minimizer not available                                                ]8;id=519478;file:///home/tobi/sw/threeML/threeML/minimizer/minimization.py\minimization.py]8;;\:]8;id=340504;file:///home/tobi/sw/threeML/threeML/minimizer/minimization.py#1345\1345]8;;\

         WARNING   PyGMO is not available                                                      ]8;id=817854;file:///home/tobi/sw/threeML/threeML/minimizer/minimization.py\minimization.py]8;;\:]8;id=539298;file:///home/tobi/sw/threeML/threeML/minimizer/minimization.py#1369\1369]8;;\

         WARNING   The cthreeML package is not installed. You will not be able to use plugins which  ]8;id=696227;file:///home/tobi/sw/threeML/threeML/__init__.py\__init__.py]8;;\:]8;id=171732;file:///home/tobi/sw/threeML/threeML/__init__.py#94\94]8;;\
                  require the C/C++ interface (currently HAWC)                                                     

         WARNING   Could not import plugin HAWCLike.py. Do you have the relative instrument         ]8;id=742440;file:///home/tobi/sw/threeML/threeML/__init__.py\__init__.py]8;;\:]8;id=92509;file:///home/tobi/sw/threeML/threeML/__init__.py#144\144]8;;\
                  software installed and configured?                                                               

In [11]:
from gammapy.datasets.map import MapDataset
from gammapy.datasets import Datasets
from gammapy.data.data_store import DataStore
from pathlib import Path

# Check package versions
import astropy.units as u
from astropy.coordinates import Angle, SkyCoord
from regions import CircleSkyRegion

# %matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from gammapy.data import DataStore
from gammapy.datasets import (
    Datasets,
    FluxPointsDataset,
    SpectrumDataset,
    SpectrumDatasetOnOff,
)
from gammapy.estimators import FluxPointsEstimator
from gammapy.estimators.utils import resample_energy_edges
from gammapy.makers import (
    ReflectedRegionsBackgroundMaker,
    SafeMaskMaker,
    SpectrumDatasetMaker,
)
from gammapy.maps import MapAxis, RegionGeom, WcsGeom
from gammapy.modeling import Fit
from gammapy.modeling.models import (
    ExpCutoffPowerLawSpectralModel,
    SkyModel,
    create_crab_spectral_model,
)
ds = DataStore.from_dir("/data/tobi/gammapy-datasets/dev/hess-dl3-dr1/")

In [12]:
from astromodels import Powerlaw
from gammapy_plugin.model_converter import SpectralModelGenerator
import astropy.units as u
import numpy as np

# Define a point source model with power law spectrum

In [13]:
spectrum = Powerlaw()
source = PointSource("crab", ra=83.63, dec=22.01 , spectral_shape=spectrum)

spectrum.piv = 1 * u.TeV
spectrum.piv.fix = True
spectrum.piv.unit = u.TeV

spectrum.K = 3.37e-11 / (u.TeV * u.cm**2 * u.s)  # norm (in 1/(TeV cm2 s))
spectrum.K.unit = 1 / (u.TeV * u.cm**2 * u.s)

#spectrum.K.fix = True

#spectrum.K.min_value = 1e-12 / (u.TeV * u.cm**2 * u.s)
#spectrum.K.max_value = 1e-10 / (u.TeV * u.cm**2 * u.s)

spectrum.index = -2.53

#testing a bunch of things
#smg = SpectralModelGenerator(spectrum)
#gpy_p = smg.class_def()

In [14]:

f_model = Model(source)
f_model.crab.spectrum.main.Powerlaw.K.prior = Log_uniform_prior(lower_bound = 10e-30,upper_bound = 10e-1)
f_model.crab.spectrum.main.Powerlaw.index.prior = Uniform_prior(lower_bound= -4,upper_bound = 4)
f_model.display()
#ene = np.array([1,10,100])


Model summary:
==============

                  N
Point sources     1
Extended sources  0
Particle sources  0

Free parameters (2):
--------------------

                                  value min_value max_value            unit
crab.spectrum.main.Powerlaw.K       0.0       0.0    1000.0  TeV-1 s-1 cm-2
crab.spectrum.main.Powerlaw.index -2.53     -10.0      10.0                

Fixed parameters (3):
(abridged. Use complete=True to see all fixed parameters)


Properties (0):
--------------------

(none)


Linked parameters (0):
----------------------

(none)

Independent variables:
----------------------

(none)

Linked functions (0):
----------------------

(none)

In [15]:
#gpy_p(ene)

In [16]:
#gpy_p


# VERITAS

In [17]:
from gammapy_plugin.GammapyLike import GammapyLike

In [18]:
VERITAS = GammapyLike("VERITAS")

In [19]:
import yaml
with open("/home/tobi/sw/gammapy-plugin/examples/config.yaml","r") as f:
    config_file = yaml.safe_load(f)
data_dir = config_file['data']['anasum']
output_dir = config_file['fileio']['outdir']
on_region_radius = Angle(
    "{} deg".format(np.sqrt(config_file['cuts']['th2cut']))
)
emin = config_file['selection']['emin']
emax = config_file['selection']['emax']
nbin = config_file['selection']['nbin']
exclusion_on = config_file['selection']['exc_on_region_radius']
exc_radius = config_file['selection']['exc_radius']
datastore = DataStore.from_dir(data_dir)
obs_table = datastore.obs_table
obs_ids = obs_table['OBS_ID']
available_irf = ["aeff", "edisp"]
observations = datastore.get_observations(
    obs_ids, required_irf=available_irf
)
RA = obs_table['RA_OBJ']
DEC = obs_table['DEC_OBJ']
RA_OBJ = RA[0]
DEC_OBJ = DEC[0]
target_position = SkyCoord(
    ra=RA_OBJ, dec=DEC_OBJ, unit="deg", frame="icrs"
)
on_region_radius = Angle("{} deg".format(np.sqrt(0.008)))
on_region = CircleSkyRegion(
    center=target_position, radius=on_region_radius
)

exclusion_mask = []

reg0 = CircleSkyRegion(
    center=SkyCoord(RA_OBJ, DEC_OBJ, unit="deg", frame="icrs"),
    radius= exclusion_on * u.deg,
)
reg1 = CircleSkyRegion(
    center=SkyCoord(81.9087, 21.937, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg2 = CircleSkyRegion(
    center=SkyCoord(82.6806, 22.4623, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg3 = CircleSkyRegion(
    center=SkyCoord(83.4118, 20.4742, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg4 = CircleSkyRegion(
    center=SkyCoord(84.1099, 21.9931, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg5 = CircleSkyRegion(
    center=SkyCoord(84.4112, 21.1425, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg6 = CircleSkyRegion(
    center=SkyCoord(84.8629, 21.7629, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg7 = CircleSkyRegion(
    center=SkyCoord(85.4782, 23.3262, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg8 = CircleSkyRegion(
    center=SkyCoord(85.5166, 22.6603, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)

skydir = target_position.galactic
geom = WcsGeom.create(
    npix=(1000, 1000),
    binsz=0.005,
    skydir=skydir,
    proj="TAN",
    frame="icrs",
)
exclusion_mask = ~geom.region_mask(
    [reg0, reg1, reg2, reg3, reg4, reg5, reg6, reg7, reg8]
)

energy_ax = MapAxis.from_energy_bounds(
    1e-2,
    1e4,
    nbin= nbin,
    per_decade=True,
    unit="TeV",
    name="energy",
)
energy_ax_true = MapAxis.from_energy_bounds(
    1e-2,
    1e4,
    nbin= nbin,
    per_decade=True,
    unit="TeV",
    name="energy_true",
)

geom = RegionGeom.create(region=on_region, axes=[energy_ax])
dataset_empty = SpectrumDataset.create(
    geom=geom, energy_axis_true=energy_ax_true
)

dataset_maker = SpectrumDatasetMaker(
    containment_correction=False,
    selection=["counts", "exposure", "edisp"],
)
bkg_maker = ReflectedRegionsBackgroundMaker(
    exclusion_mask=exclusion_mask
)

safe_mask_masker = SafeMaskMaker(
    methods=["aeff-max"], aeff_percent=10
)

datasets = Datasets()
for obs_id, observation in zip(obs_ids, observations):
    dataset = dataset_maker.run(
        dataset_empty.copy(name=str(obs_id)), observation
    )
    dataset_on_off = bkg_maker.run(dataset, observation)
    dataset_on_off = safe_mask_masker.run(
        dataset_on_off, observation
    )
    datasets.append(dataset_on_off)


In [21]:
VERITAS.set_datasets(datasets)

In [22]:
veritas_data = DataList(VERITAS)

In [23]:
# If checking just the veritas fit, use this
# jl = JointLikelihood(f_model, veritas_data, verbose=True)

# FERMI

In [24]:
lat_catalog = FermiLATSourceCatalog()

ra, dec, table = lat_catalog.search_around_source("Crab", radius=0.004)

table

Trying https://heasarc.gsfc.nasa.gov/cgi-bin/vo/cone/coneGet.pl?table=fermilpsc&


name,source_type,short_source_type,ra,dec,assoc_name,tevcat_assoc,Search_Offset
,,,deg,deg,,,
object,str18,object,float64,float64,object,object,float64
4FGL J0534.5+2201s,pulsar wind nebula,PWN,83.6331,22.0199,Crab Nebula,Crab,0.1552
4FGL J0534.5+2201i,pulsar wind nebula,PWN,83.6330,22.0200,Crab Nebula,Crab,0.1597


In [25]:
model = lat_catalog.get_model()

12:22:14 WARNING   We have set the min_value of Crab_synch.spectrum.main.Log_parabola.K to 1e-99   ]8;id=794166;file:///home/tobi/sw/astromodels/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=122359;file:///home/tobi/sw/astromodels/astromodels/core/parameter.py#704\704]8;;\
                  because there was a postive transform                                                            

         WARNING   We have set the min_value of Crab_synch.spectrum.main.Log_parabola.K to 1e-99   ]8;id=633234;file:///home/tobi/sw/astromodels/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=475164;file:///home/tobi/sw/astromodels/astromodels/core/parameter.py#704\704]8;;\
                  because there was a postive transform                                                            

12:22:14 WARNING   Source 4FGL J0534.5+2201i is extended, but morphology information is             ]8;id=551556;file:///home/tobi/sw/threeML/threeML/catalogs/FermiLAT.py\FermiLAT.py]8;;\:]8;id=446452;file:///home/tobi/sw/threeML/threeML/catalogs/FermiLAT.py#208\208]8;;\
                  unavailable. I will provide a point source instead                                               

         WARNING   We have set the min_value of Crab_IC.spectrum.main.Log_parabola.K to 1e-99      ]8;id=378068;file:///home/tobi/sw/astromodels/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=735213;file:///home/tobi/sw/astromodels/astromodels/core/parameter.py#704\704]8;;\
                  because there was a postive transform                                                            

         WARNING   We have set the min_value of Crab_IC.spectrum.main.Log_parabola.K to 1e-99      ]8;id=724683;file:///home/tobi/sw/astromodels/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=770569;file:///home/tobi/sw/astromodels/astromodels/core/parameter.py#704\704]8;;\
                  because there was a postive transform                                                            

In [26]:
model.free_point_sources_within_radius(3.0, normalization_only=True)
model.Crab_IC.spectrum.main.Log_parabola.K.prior = Log_uniform_prior(lower_bound =10**(-3),upper_bound = 10**3)
model.Crab_synch.spectrum.main.Log_parabola.K.prior = Log_uniform_prior(lower_bound =10**(-3),upper_bound = 10**3)


model.display()

Model summary:
==============

                  N
Point sources     2
Extended sources  0
Particle sources  0

Free parameters (2):
--------------------

                                        value min_value max_value  \
Crab_synch.spectrum.main.Log_parabola.K   0.0       0.0       0.0   
Crab_IC.spectrum.main.Log_parabola.K      0.0       0.0       0.0   

                                                   unit  
Crab_synch.spectrum.main.Log_parabola.K  keV-1 s-1 cm-2  
Crab_IC.spectrum.main.Log_parabola.K     keV-1 s-1 cm-2  

Fixed parameters (10):
(abridged. Use complete=True to see all fixed parameters)


Properties (0):
--------------------

(none)


Linked parameters (0):
----------------------

(none)

Independent variables:
----------------------

(none)

Linked functions (0):
----------------------

(none)

In [27]:
ra = 83.63
dec = 22.01

In [28]:
# Download data from Jan 01 2010 to February 1 2010

tstart = "2010-01-01 00:00:00"
tstop = "2010-02-01 00:00:00"

# Note that this will understand if you already download these files, and will
# not do it twice unless you change your selection or the outdir

evfile, scfile = download_LAT_data(
    ra,
    dec,
    20.0,
    tstart,
    tstop,
    time_type="Gregorian",
    destination_directory="Crab_data",
)

12:22:16 INFO      Query parameters:                                                       ]8;id=420;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py\download_LAT_data.py]8;;\:]8;id=412464;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py#262\262]8;;\

         INFO                          coordfield = 83.6300,22.0100                        ]8;id=101634;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py\download_LAT_data.py]8;;\:]8;id=908066;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py#266\266]8;;\

         INFO                         coordsystem = J2000                                  ]8;id=801295;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py\download_LAT_data.py]8;;\:]8;id=376285;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py#266\266]8;;\

         INFO                          shapefield = 20.0                                   ]8;id=43115;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py\download_LAT_data.py]8;;\:]8;id=372908;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py#266\266]8;;\

         INFO                           timefield = 2010-01-01 00:00:00,2010-02-01         ]8;id=967824;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py\download_LAT_data.py]8;;\:]8;id=299832;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py#266\266]8;;\
                  00:00:00                                                                                         

...

         INFO                         destination = query                                  ]8;id=941123;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py\download_LAT_data.py]8;;\:]8;id=889697;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py#266\266]8;;\

         INFO                          spacecraft = checked                                ]8;id=676208;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py\download_LAT_data.py]8;;\:]8;id=715374;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py#266\266]8;;\

         INFO      Query ID: e94fe495b65679baa926097eebf7f7fd                              ]8;id=335573;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py\download_LAT_data.py]8;;\:]8;id=791372;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py#271\271]8;;\

         WARNING   Existing event file                                                     ]8;id=247637;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py\download_LAT_data.py]8;;\:]8;id=846757;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py#316\316]8;;\
                  [PosixPath('/home/tobi/sw/gammapy-plugin/notebooks/Crab_data/L2412020725                         
                  0949176D7595_PH00.fits')] and Spacecraft file                                                    
                  /home/tobi/sw/gammapy-plugin/notebooks/Crab_data/L24120207233249176D7524                         
                  _SC00.fits correspond to the same selection. We assume you did not                               
                  tamper with them, so we will return those instead of downloading them                            
                  again. If you want to download them again, remove them from the outdir                           

         WARNING   Existing merged event file                                               ]8;id=189619;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py\download_LAT_data.py]8;;\:]8;id=814917;file:///home/tobi/sw/threeML/threeML/utils/data_download/Fermi_LAT/download_LAT_data.py#88\88]8;;\
                  /home/tobi/sw/gammapy-plugin/notebooks/Crab_data/Le94fe495b65679baa926097                        
                  eebf7f7fd_FT1.fits correspond to the same selection. We assume you did                           
                  not tamper with it, so we will return it instead of merging it again. If                         
                  you want to redo the FT1 file again, remove it from the outdir                                   

In [29]:
import fermipy

In [30]:
config = FermipyLike.get_basic_config(
    evfile=evfile,
    scfile=scfile,
    ra=ra,
    dec=dec,
    fermipy_verbosity=1,
    fermitools_chatter=0,
)

# See what we just got

config.display()

binning:
  binsperdec: 8
  binsz: 0.1
  roiwidth: 10.0
data:
  evfile: /home/tobi/sw/gammapy-plugin/notebooks/Crab_data/Le94fe495b65679baa926097eebf7f7fd_FT1.fits
  scfile: /home/tobi/sw/gammapy-plugin/notebooks/Crab_data/L24120207233249176D7524_SC00.fits
logging:
  chatter: 0
  verbosity: 1
selection:
  dec: 22.01
  emax: 100000.0
  emin: 100.0
  evclass: 128
  evtype: 3
  filter: DATA_QUAL>0 && LAT_CONFIG==1
  ra: 83.63
  tmax: 286675202.0
  tmin: 283996802.0
  zmax: 100.0



binning:
 binsperdec: 8
 binsz: 0.1
 roiwidth: 10.0
data:
 evfile: /home/tobi/sw/gammapy-plugin/notebooks/Crab_data/Le94fe495b65679baa926097eebf7f7fd_FT1.fits
 scfile: /home/tobi/sw/gammapy-plugin/notebooks/Crab_data/L24120207233249176D7524_SC00.fits
logging:
 chatter: 0
 verbosity: 1
selection:
 dec: 22.01
 emax: 100000.0
 emin: 100.0
 evclass: 128
 evtype: 3
 filter: DATA_QUAL>0 && LAT_CONFIG==1
 ra: 83.63
 tmax: 286675202.0
 tmin: 283996802.0
 zmax: 100.0

In [31]:
LAT = FermipyLike("LAT", config)

config.display()

binning:
  binsperdec: 8
  binsz: 0.1
  roiwidth: 10.0
data:
  evfile: /home/tobi/sw/gammapy-plugin/notebooks/Crab_data/Le94fe495b65679baa926097eebf7f7fd_FT1.fits
  scfile: /home/tobi/sw/gammapy-plugin/notebooks/Crab_data/L24120207233249176D7524_SC00.fits
fileio:
  outdir: __8a152570c38eb6d74f9fe58e8079f9dd
logging:
  chatter: 0
  verbosity: 1
selection:
  dec: 22.01
  emax: 100000.0
  emin: 100.0
  evclass: 128
  evtype: 3
  filter: DATA_QUAL>0 && LAT_CONFIG==1
  ra: 83.63
  tmax: 286675202.0
  tmin: 283996802.0
  zmax: 100.0



binning:
 binsperdec: 8
 binsz: 0.1
 roiwidth: 10.0
data:
 evfile: /home/tobi/sw/gammapy-plugin/notebooks/Crab_data/Le94fe495b65679baa926097eebf7f7fd_FT1.fits
 scfile: /home/tobi/sw/gammapy-plugin/notebooks/Crab_data/L24120207233249176D7524_SC00.fits
fileio:
 outdir: __8a152570c38eb6d74f9fe58e8079f9dd
logging:
 chatter: 0
 verbosity: 1
selection:
 dec: 22.01
 emax: 100000.0
 emin: 100.0
 evclass: 128
 evtype: 3
 filter: DATA_QUAL>0 && LAT_CONFIG==1
 ra: 83.63
 tmax: 286675202.0
 tmin: 283996802.0
 zmax: 100.0

# Joint fit 

In [ ]:
data = DataList(VERITAS, LAT)

jl = JointLikelihood(f_model, data,)
f_model.display()

12:22:30 INFO      Using IRFs P8R3_SOURCE_V3                                                     ]8;id=665392;file:///home/tobi/sw/threeML/threeML/plugins/FermipyLike.py\FermipyLike.py]8;;\:]8;id=129906;file:///home/tobi/sw/threeML/threeML/plugins/FermipyLike.py#126\126]8;;\


Found Galactic template for IRF. P8R3_SOURCE_V3: /home/tobi/bin/anaconda3/envs/fermipy/share/fermitools/refdata/fermi/galdiffuse/gll_iem_v07.fits

Cutting the template around the ROI: 


Found Isotropic template for irf P8R3_SOURCE_V3: /home/tobi/bin/anaconda3/envs/fermipy/share/fermitools/refdata/fermi/galdiffuse/iso_P8R3_SOURCE_V3_v1.txt


In [ ]:
res = jl.fit()



In [ ]:
JF_result = jl.results

In [ ]:
JF_result.display()

In [ ]:
res = jl.get_errors()

# Some plots

In [ ]:
res = jl.get_contours(
    f_model.crab.spectrum.main.Powerlaw.index, -2.5, -2, 30
)

In [ ]:
res = jl.get_contours(
    f_model.crab.spectrum.main.Powerlaw.K, 1,6, 30
)

In [ ]:
res = jl.get_contours(
    "crab.spectrum.main.Powerlaw.K", 1, 6, 30,
    "crab.spectrum.main.Powerlaw.index",
    -2.5, -2, 30)

In [ ]:
fluxes = jl.results.get_flux(100 * u.keV, 1 * u.MeV)

# Same results would be obtained with
# fluxes = results_reloaded.get_point_source_flux(100 * u.keV, 1 * u.MeV)

In [ ]:
fluxes["flux"].values[0]

In [ ]:
plot_spectra(jl.results, ene_min=0.1, ene_max=1e6, num_ene=500, 
                          flux_unit='erg / (cm2 s)')

In [ ]:
JF_result.optimized_model.crab.spectrum.main.Powerlaw.K.value/(10**9)

In [ ]:
JF_result.optimized_model.crab.spectrum.main.Powerlaw.K

In [ ]:
VERITAS.